In [1]:
#import kagglehub

#Download latest version
#path = kagglehub.dataset_download("dhoogla/csecicids2018")

#print("Path to dataset files:", path)


In [2]:
import os
import gc
import pandas as pd

dataset_path = '/kaggle/input/datasets/dhoogla/csecicids2018'

files = [
    os.path.join(dataset_path,file)
    for file in os.listdir(dataset_path)
    if file.endswith(".parquet")
]
datanot = []
for i in files:
    df = pd.read_parquet(i)
    cols_to_drop = ['Timestamp', 'Dst Port', 'Src IP', 'Dst IP', 'Src Port']
    df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = df[col].astype('int32')
    datanot.append(df)
    
data = pd.concat(datanot, ignore_index=True)
del datanot
gc.collect()
print("ouiiiii")

ouiiiii


In [3]:
print(data.columns)

Index(['Protocol', 'Flow Duration', 'Total Fwd Packets',
       'Total Backward Packets', 'Fwd Packets Length Total',
       'Bwd Packets Length Total', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Min', 'Bwd Packet Length Mean',
       'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
       'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
       'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
       'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std',
       'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags',
       'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
       'Packet Length Min', 'Packet Length Max', 'Packet Length Mean',
       'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count',
       'SYN Flag Count', 'RST Fla

In [4]:
#print(data.columns)
dont_need = [
    'Protocol',
    'Fwd Packet Length Min',
    'Fwd Packet Length Std',
    'Bwd Packet Length Min',
    'Bwd Packet Length Std',
    'Flow Packets/s',
    'Flow IAT Max',
    'Flow IAT Min',
    'Fwd IAT Mean',
    'Fwd IAT Max',
    'Fwd IAT Min',
    'Bwd IAT Total',
    'Bwd IAT Mean',
    'Bwd IAT Std',
    'Bwd IAT Max',
    'Bwd IAT Min',
    'Fwd PSH Flags',
    'Bwd PSH Flags',
    'Fwd URG Flags',
    'Bwd URG Flags',
    'Bwd Header Length',
    'Packet Length Min',
    'Packet Length Max',
    'Packet Length Mean',
    'Packet Length Variance',
    'URG Flag Count',
    'CWE Flag Count',
    'ECE Flag Count',
    'Avg Fwd Segment Size',
    'Avg Bwd Segment Size',
    'Fwd Avg Bytes/Bulk',
    'Fwd Avg Packets/Bulk',
    'Fwd Avg Bulk Rate',
    'Bwd Avg Bytes/Bulk',
    'Bwd Avg Packets/Bulk',
    'Bwd Avg Bulk Rate',
    'Subflow Fwd Packets',
    'Subflow Fwd Bytes',
    'Subflow Bwd Packets',
    'Subflow Bwd Bytes',
    'Fwd Act Data Packets',
    'Active Std',
    'Active Max',
    'Active Min',
    'Idle Std',
    'Idle Max',
    'Idle Min'
]
data = data.drop(columns = dont_need)
familles = {
    'Benign': 'Benign',
    'Brute Force -Web': 'Web Attack',
    'Brute Force -XSS': 'Web Attack',
    'SQL Injection': 'Web Attack',
    'FTP-BruteForce': 'BruteForce',
    'SSH-Bruteforce': 'BruteForce',
    'DoS attacks-Hulk': 'DoS',
    'DoS attacks-GoldenEye': 'DoS',
    'DoS attacks-Slowloris': 'DoS',
    'DoS attacks-SlowHTTPTest': 'DoS',
    'DDOS attack-HOIC': 'DDoS',
    'DDoS attacks-LOIC-HTTP': 'DDoS',
    'DDOS attack-LOIC-UDP': 'DDoS',
    'Bot': 'Botnet_Infiltration',
    'Infilteration': 'Botnet_Infiltration'
}
data['Label'] = data['Label'].map(familles)
#print(data.columns)

In [5]:
#X = data.drop(columns = 'Label')
#yTmp = data['Label']
#print("X............")
#print("\nX............",X.columns)
#print("\nY............")
#print("\n",yTmp.name)

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

X = data.drop(columns = 'Label')
yTmp = data['Label']
le = LabelEncoder()
y = le.fit_transform(yTmp)
X_train,X_test,y_train,y_test = train_test_split(
    X,y,
    random_state = 49, #haha my fav number squared
    test_size = 0.2,
    shuffle=True,
    stratify = y
)
print('yea playa')

yea playa


In [7]:
tmp_data = X_train.copy()
tmp_data['Label'] = y_train
data_attack = tmp_data[tmp_data['Label'] != 0].copy()
data_normal = tmp_data[tmp_data['Label'] == 0].copy()
data_normal = data_normal.drop_duplicates()
data_unsup = data_normal.copy()
data_normal = data_normal.sample(n=1_500_000, random_state=49)
data_pro = pd.concat([data_attack,data_normal])
#print(data_pro.tail())
#print(data_normal.head())
#print(data_attack.head())
#print(tmp_data.head())

In [8]:
y_train_pro = data_pro['Label']
X_train_pro = data_pro.drop(columns = 'Label')
#print(X_train_pro.columns)
#print(y_train_pro.head())

In [9]:
#print(X_train.head())
import numpy as np
print("Valeurs NaN :", data.isna().sum().sum())
print("Valeurs infinies :", np.isinf(data.select_dtypes(include=[np.number])).sum().sum())
data.info()

Valeurs NaN : 0
Valeurs infinies : 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6659532 entries, 0 to 6659531
Data columns (total 31 columns):
 #   Column                    Dtype  
---  ------                    -----  
 0   Flow Duration             int32  
 1   Total Fwd Packets         int32  
 2   Total Backward Packets    int32  
 3   Fwd Packets Length Total  int32  
 4   Bwd Packets Length Total  float64
 5   Fwd Packet Length Max     int32  
 6   Fwd Packet Length Mean    float32
 7   Bwd Packet Length Max     int32  
 8   Bwd Packet Length Mean    float32
 9   Flow Bytes/s              float32
 10  Flow IAT Mean             float32
 11  Flow IAT Std              float32
 12  Fwd IAT Total             float64
 13  Fwd IAT Std               float32
 14  Fwd Header Length         int32  
 15  Fwd Packets/s             float32
 16  Bwd Packets/s             float32
 17  Packet Length Std         float32
 18  FIN Flag Count            int8   
 19  SYN Flag Count            

In [10]:
from sklearn.pipeline import Pipeline
from lightgbm import LGBMClassifier as lgbm, early_stopping
from sklearn.metrics import classification_report
import numpy as np

tab = y_train_pro.value_counts()
n = tab[0]
PoidMax = 50.0
poids = {}
for cls, packs in tab.items():
    if cls == 0:
        poids[cls] = 1.0
    else:
        i = np.sqrt(n/packs)
        poids[cls] = float(min(PoidMax,i))


my_pipa = Pipeline(steps=[ #chb4at tkon had pipa hh
    ('classifier', lgbm(
        n_estimators=1000,
        learning_rate = 0.05,
        random_state=49,
        n_jobs=-1,
        class_weight = poids
    ))
])
X_train_train,X_train_stop,y_train_train,y_train_stop = train_test_split(
    X_train_pro,y_train_pro,
     random_state = 49,
    test_size = 0.10,
    stratify=y_train_pro
)
my_pipa.fit(X_train_train,y_train_train,classifier__eval_set=[(X_train_stop, y_train_stop)],
    classifier__callbacks=[early_stopping(stopping_rounds=30)])

probs = my_pipa.predict_proba(X_test)
y_pred = np.where(probs.max(axis=1) >= 0.85, probs.argmax(axis=1), 0)

dika = (y_pred == y_test).mean()
print(f"the dika is here: {dika * 100:.2f}%")
print(classification_report(
    y_test, y_pred,
    target_names=le.classes_, 
    zero_division=0
))

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.180393 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5825
[LightGBM] [Info] Number of data points in the train set: 2307977, number of used features: 30
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Start training from score -0.950627
[LightGBM] [Info] Start training from score -1.932697
[LightGBM] [Info] Start training from score -2.446622
[LightGBM] [Info] Start training from score -1.391763
[LightGBM] [Info] Start training from score -2.078303
[LightGBM] [Info] Start training from score -4.781937
Training until validation scores don't improve for 30 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

In [11]:
max_probs = probs.max(axis=1)
preds = probs.argmax(axis=1)

bot_inf_idx = int(le.transform(['Botnet_Infiltration'])[0])
web_idx = int(le.transform(['Web Attack'])[0])

thresholds = np.full(len(preds), 0.85)
thresholds[preds == bot_inf_idx] = 0.90
thresholds[preds == web_idx] = 0.90

y_pred = np.where(max_probs >= thresholds, preds, 0)

dika = (y_pred == y_test).mean()
print(f"the dika is here: {dika * 100:.2f}%")
print(classification_report(
    y_test, y_pred,
    target_names=le.classes_,
    zero_division=0
))

the dika is here: 98.16%
                     precision    recall  f1-score   support

             Benign       0.98      1.00      0.99   1065802
Botnet_Infiltration       0.97      0.55      0.70     52604
         BruteForce       1.00      1.00      1.00     18820
               DDoS       1.00      1.00      1.00    155191
                DoS       1.00      1.00      1.00     39314
         Web Attack       0.84      0.70      0.77       176

           accuracy                           0.98   1331907
          macro avg       0.96      0.88      0.91   1331907
       weighted avg       0.98      0.98      0.98   1331907



In [12]:
lgbm_model = my_pipa.named_steps['classifier']

importance_df = pd.DataFrame({
    'Feature': X_train_pro.columns,
    'Gain': lgbm_model.booster_.feature_importance(importance_type='gain')
}).sort_values(by='Gain', ascending=False)

print(importance_df.head(20))

                     Feature          Gain
14         Fwd Header Length  1.835835e+07
25        Init Fwd Win Bytes  1.767013e+07
27          Fwd Seg Size Min  1.560660e+07
5      Fwd Packet Length Max  1.304874e+07
6     Fwd Packet Length Mean  6.476679e+06
10             Flow IAT Mean  5.659903e+06
20            RST Flag Count  4.557227e+06
0              Flow Duration  2.936943e+06
26        Init Bwd Win Bytes  2.147496e+06
9               Flow Bytes/s  1.974844e+06
13               Fwd IAT Std  1.591340e+06
17         Packet Length Std  1.574014e+06
1          Total Fwd Packets  1.345878e+06
15             Fwd Packets/s  1.206976e+06
12             Fwd IAT Total  1.159797e+06
7      Bwd Packet Length Max  8.008399e+05
29                 Idle Mean  7.494834e+05
2     Total Backward Packets  6.868817e+05
3   Fwd Packets Length Total  6.188388e+05
24           Avg Packet Size  5.910115e+05


In [13]:
!pip install -q onnxmltools skl2onnx
import json
import numpy as np
from onnxmltools.convert import convert_lightgbm
from onnxmltools.convert.common.data_types import FloatTensorType

booster = my_pipa.named_steps['classifier']
n_features = X_train_pro.shape[1]

onnx_model = convert_lightgbm(
    booster,
    initial_types=[('float_input', FloatTensorType([None, n_features]))],
    target_opset=15,
    zipmap=False
)

with open('ids_lightgbm.onnx', 'wb') as f:
    f.write(onnx_model.SerializeToString())

class_thresholds = np.full(len(le.classes_), 0.85)
class_thresholds[int(le.transform(['Botnet_Infiltration'])[0])] = 0.90
class_thresholds[int(le.transform(['Web Attack'])[0])] = 0.90

metadata = {
    'feature_names': list(X_train_pro.columns),
    'classes': le.classes_.tolist(),
    'thresholds': class_thresholds.tolist(),
    'benign_idx': int(le.transform(['Benign'])[0])
}

with open('ids_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Saved ids_lightgbm.onnx and ids_metadata.json")

Saved ids_lightgbm.onnx and ids_metadata.json


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report
import numpy as np

y_normal = data_normal['Label']
X_normal = np.log1p(np.maximum(data_normal.drop(columns='Label').astype(np.float32), np.float32(0.0)))

X_train_normal, X_test_normal, y_train_normal, y_test_normal = train_test_split(
    X_normal, y_normal,
    random_state=49, #haha my fav number squared
    test_size=0.2,
    shuffle=True,
    stratify=y_normal
)

my_pipa_unsup = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('detector', IsolationForest(
        n_estimators=250,
        max_samples=2048,
        contamination='auto',
        max_features=1.0,
        bootstrap=False,
        n_jobs=-1,
        random_state=49
    ))
])

my_pipa_unsup.fit(X_train_normal)

normal_scores = my_pipa_unsup.decision_function(X_test_normal)

threshold = np.percentile(normal_scores, 0.5)

X_test_scaled = np.log1p(np.maximum(X_test.astype(np.float32), np.float32(0.0)))
test_scores = my_pipa_unsup.decision_function(X_test_scaled)

y_pred_unsup = (test_scores < threshold).astype(int)

y_test_binary = (y_test != 'Benign').astype(int) if y_test.dtype == 'object' else (y_test != 0).astype(int)

print(classification_report(
    y_test_binary,
    y_pred_unsup,
    target_names=['Benign', 'Attack'],
    zero_division=0
))

              precision    recall  f1-score   support

      Benign       0.80      0.99      0.89   1065802
      Attack       0.22      0.01      0.01    266105

    accuracy                           0.80   1331907
   macro avg       0.51      0.50      0.45   1331907
weighted avg       0.69      0.80      0.71   1331907



In [15]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_test_binary, -test_scores)
print(f"Unsupervised ROC-AUC: {auc:.4f}")
print("Benign score percentiles (0.1%, 1%, 5%, 50%):", np.percentile(test_scores[y_test_binary == 0], [0.1, 1, 5, 50]))
print("Attack score percentiles (10%, 50%, 90%):    ", np.percentile(test_scores[y_test_binary == 1], [10, 50, 90]))

Unsupervised ROC-AUC: 0.8620
Benign score percentiles (0.1%, 1%, 5%, 50%): [-0.16560109 -0.11342689 -0.04890499  0.07271074]
Attack score percentiles (10%, 50%, 90%):     [-0.05041961 -0.02565294  0.04002349]


In [16]:
!pip install -q onnxmltools skl2onnx onnxruntime
import json
import numpy as np
import onnxruntime as ort
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

n_features = X_train_normal.shape[1]
initial_type = [('float_input', FloatTensorType([None, n_features]))]

onnx_model = convert_sklearn(
    my_pipa_unsup,
    initial_types=initial_type,
    target_opset={'': 17, 'ai.onnx.ml': 3}
)

onnx_path = "isolation_forest.onnx"
with open(onnx_path, "wb") as f:
    f.write(onnx_model.SerializeToString())

session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
input_name = session.get_inputs()[0].name
output_names = [out.name for out in session.get_outputs()]

sample = X_test_normal.iloc[:5].to_numpy(dtype=np.float32)
sk_scores = my_pipa_unsup.decision_function(sample)
ort_scores = session.run(None, {input_name: sample})[1].ravel()

print("Outputs:", output_names)
print("Max diff (sklearn vs ONNX):", np.max(np.abs(sk_scores - ort_scores)))

metadata = {
    "model_file": onnx_path,
    "input_name": input_name,
    "score_output_name": output_names[1],
    "threshold": float(threshold),
    "anomaly_condition": "score < threshold",
    "preprocessing": "log1p(max(x, 0.0f))",
    "num_features": int(n_features),
    "features": list(X_train_normal.columns)
}

with open("unsup_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Saved unsup_metadata.json successfully.")

Outputs: ['label', 'scores']
Max diff (sklearn vs ONNX): 8.727468858715781e-08
Saved unsup_metadata.json successfully.


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
